In [ ]:
%pip install ultralytics roboflow opencv-python matplotlib

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="")  # ← paste your key here

project = rf.workspace("hammer-dqemv").project("tools-vl4du")
dataset = project.version(6).download("yolov8")

print("✅ Dataset downloaded to:", dataset.location)

In [ ]:
%ls
%cd Tools-6/
%ls

In [ ]:
# Check what's inside data.yaml
with open("data.yaml", "r") as f:
    print(f.read())

In [ ]:
import yaml
import os

# Current working directory
current_dir = os.getcwd()  # should be /content/Tools-6

# Fix the yaml
with open("data.yaml", "r") as f:
    data = yaml.safe_load(f)

# Update paths to absolute
data['train'] = os.path.join(current_dir, "train/images")
data['val']   = os.path.join(current_dir, "valid/images")
data['test']  = os.path.join(current_dir, "test/images")

# Save back
with open("data.yaml", "w") as f:
    yaml.dump(data, f)

print("✅ data.yaml updated!")
print(f"  train → {data['train']}")
print(f"  val   → {data['val']}")
print(f"  test  → {data['test']}")
print(f"  classes ({data['nc']}): {data['names']}")

In [ ]:
!pip install ultralytics -q

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # downloads pretrained weights automatically

model.train(
    data="/content/Tools-6/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="tools_detector_v1",
    device=0 if torch.cuda.is_available() else "cpu",
    patience=10,
    save=True,
    plots=True
)

print("✅ Training Done!")

In [ ]:
import os

# Search for results.png automatically
for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "results.png":
            print("✅ Found at:", os.path.join(root, file))

In [ ]:
from IPython.display import Image

# Use the correct path found above
Image("/content/Tools-6/runs/detect/tools_detector_v1/results.png", width=900)

In [ ]:
import os

save_dir = "/content/Tools-6/runs/detect/tools_detector_v1"

print("📁 Files saved in training folder:")
for f in sorted(os.listdir(save_dir)):
    print(f"   {f}")


In [ ]:
import pandas as pd

df = pd.read_csv("/content/Tools-6/runs/detect/tools_detector_v1/results.csv")
df.columns = df.columns.str.strip()  # remove whitespace

print("📊 Final Training Metrics:")
print(df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'train/box_loss', 'train/cls_loss']].to_string())

# Best epoch
best_row = df.loc[df['metrics/mAP50(B)'].idxmax()]
print(f"\n🏆 Best Epoch: {int(best_row['epoch'])}")
print(f"   mAP50:    {best_row['metrics/mAP50(B)']:.3f}")
print(f"   mAP50-95: {best_row['metrics/mAP50-95(B)']:.3f}")

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob

# Load best weights
model = YOLO("/content/Tools-6/runs/detect/tools_detector_v1/weights/best.pt")

# Test on validation images
test_imgs = glob.glob("/content/Tools-6/valid/images/*.jpg")[:3]

for img in test_imgs:
    results = model.predict(img, conf=0.4, save=True,
                           project="/content/test_results")
    for box in results[0].boxes:
        label = model.names[int(box.cls)]
        conf  = float(box.conf)
        print(f"  🔧 {label}: {conf:.0%}")
    print("---")

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/Tools-6/runs/detect/tools_detector_v1/weights/best.pt")

# Export for different uses
model.export(format="onnx")       # universal format
model.export(format="tflite")     # for mobile/Raspberry Pi

In [ ]:
import shutil
from google.colab import files

# Zip the entire weights folder
shutil.make_archive(
    "/content/tools_detector_all_weights",  # output zip name
    "zip",
    "/content/Tools-6/runs/detect/tools_detector_v1/weights"  # folder to zip
)

files.download("/content/tools_detector_all_weights.zip")